# Steam Trajectory — Scrape & Database Load

Reads the candidate list frozen by `00_cohort_selection.ipynb`, scrapes SteamCharts for each candidate's monthly history, drops any that fail, freezes the resulting **final cohort** to `cohort_appids.csv` (so downstream notebooks never depend on live network conditions again), and loads everything into `steam_project.db`.

This notebook is standalone: it never re-derives the cohort from the Kaggle metadata, and never needs to reload the full ~126K-game JSON dataset.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

/Users/pmacias/Dropbox/steamproject


In [2]:
import pandas as pd

from steam_trajectory.ingest.steamcharts_scraper import SteamChartsScraper
from steam_trajectory.ingest.kaggle_loader import KaggleLoader
from steam_trajectory.db.connection import get_connection
from steam_trajectory.db.writer import DatabaseWriter

In [3]:
candidates = pd.read_csv("candidate_appids.csv")
print(f"Loaded {len(candidates)} candidates")
candidates.head()

Loaded 382 candidates


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres,Peak CCU,Estimated owners
0,1517290,Battlefield™ 2042,"Nov 19, 2021",DICE,Electronic Arts,59.99,297774,46.742160,"Action, Adventure, Casual",2774,10000000 - 20000000
1,1088850,Marvel's Guardians of the Galaxy,"Oct 26, 2021",Eidos-Montréal,Eidos Interactive Corp.,8.99,39158,94.123806,"Action, Adventure",219,500000 - 1000000
2,1296830,暖雪 Warm Snow,"Jan 18, 2022",BadMudStudio,bilibili,12.59,37007,93.074283,"Action, Adventure, Indie, RPG",516,1000000 - 2000000
3,1170950,Mortal Online 2,"Jan 25, 2022",Star Vault AB,Star Vault AB,12.79,12943,60.805068,"Action, Indie, Massively Multiplayer, RPG",440,500000 - 1000000
4,1353300,Idle Slayer,"Dec 21, 2020",Pablo Leban,Pablo Leban,0.00,8767,85.411201,"Casual, Indie, Strategy, Free To Play",1300,500000 - 1000000


## Quick scraper sanity check
Optional — a fast check that the scraper still works before committing to the full batch. Safe to skip on reruns once you trust it.

In [4]:
scraper = SteamChartsScraper()
test_appids = candidates["AppID"].head(3).tolist()
print("Testing:", test_appids)

test_histories, test_failures = scraper.get_monthly_history_batch(test_appids)
print(f"\n{len(test_histories)} monthly records, {len(test_failures)} failures")
print(test_failures)


Testing: [1517290, 1088850, 1296830]

166 monthly records, 0 failures
[]


## Full scrape + freeze final cohort
Scrapes every candidate, drops any that fail (no SteamCharts page, timeout, etc. — see notes in `steamcharts_scraper.py`), and **freezes the resulting cohort to `cohort_appids.csv`**. This is the fix for the reproducibility gap we hit earlier: since scraping failures depend partly on live network/server conditions, rerunning this cell can legitimately produce a different-sized survivor set each time. Freezing the result means every notebook from here on reads a fixed list, rather than silently re-deriving a possibly-different cohort on every rerun.

In [5]:
all_appids = candidates["AppID"].tolist()
all_histories, failures = scraper.get_monthly_history_batch(all_appids)

print(f"\nDone. {len(all_histories)} monthly records, {len(failures)} candidates failed.")
print(failures)

Processed 20/382 games (0 failures so far)...
Processed 40/382 games (0 failures so far)...
Processed 60/382 games (0 failures so far)...
Processed 80/382 games (0 failures so far)...
Processed 100/382 games (0 failures so far)...
Processed 120/382 games (0 failures so far)...
Processed 140/382 games (0 failures so far)...
Processed 160/382 games (0 failures so far)...
Processed 180/382 games (0 failures so far)...
Processed 200/382 games (0 failures so far)...
Processed 220/382 games (1 failures so far)...
Processed 240/382 games (1 failures so far)...
Processed 260/382 games (1 failures so far)...
Processed 280/382 games (1 failures so far)...
Processed 300/382 games (1 failures so far)...
Processed 320/382 games (2 failures so far)...
Processed 340/382 games (2 failures so far)...
Processed 360/382 games (2 failures so far)...
Processed 380/382 games (2 failures so far)...

Done. 27549 monthly records, 2 candidates failed.
[{'appid': 774171, 'error': '404 Client Error: Not Found for

In [6]:
failed_appids = {f["appid"] for f in failures}
final_cohort = candidates[~candidates["AppID"].isin(failed_appids)].reset_index(drop=True)

final_appids = set(final_cohort["AppID"])
final_histories = [record for record in all_histories if record["appid"] in final_appids]

print(f"Final cohort size: {len(final_cohort)}")
print(f"Monthly records for final cohort: {len(final_histories)}")

final_cohort.to_csv("cohort_appids.csv", index=False)
print("Saved final cohort to cohort_appids.csv")

Final cohort size: 380
Monthly records for final cohort: 27549
Saved final cohort to cohort_appids.csv


## Load into the database
`KaggleLoader.iter_games` / `iter_genres` are called as staticmethods here — no `KaggleLoader` instance needed, so this never reloads the full JSON dataset.

In [7]:
conn = get_connection("steam_project.db")
writer = DatabaseWriter(conn)

# Clear all three tables tied to cohort membership before reloading —
# order matters due to foreign key constraints: monthly_metrics and
# game_genres both reference games.appid, so they must be cleared
# before games itself. (genres, the lookup table, doesn't need
# clearing — genre names are reused across cohort versions, not
# tied to any specific cohort.)
conn.execute("DELETE FROM monthly_metrics")
conn.execute("DELETE FROM game_genres")
conn.execute("DELETE FROM games")
conn.commit()

# 1. Games
for game in KaggleLoader.iter_games(final_cohort):
    writer.insert_game(**game)

# 2. Genre links
for appid, genre_name in KaggleLoader.iter_genres(final_cohort):
    writer.link_game_genre(appid, genre_name)

# 3. Monthly metrics from the scrape
for record in final_histories:
    writer.insert_monthly_metric(**record)

writer.commit()
print("Database load complete.")

Database load complete.


## Verify the load

In [8]:
print("games:", pd.read_sql("SELECT COUNT(*) as n FROM games", conn)["n"][0])
print("game_genres:", pd.read_sql("SELECT COUNT(*) as n FROM game_genres", conn)["n"][0])
print("monthly_metrics:", pd.read_sql("SELECT COUNT(*) as n FROM monthly_metrics", conn)["n"][0])

pd.read_sql("""
    SELECT g.name, g.release_date, gen.name AS genre, COUNT(m.month) AS months_tracked
    FROM games g
    JOIN game_genres gg ON g.appid = gg.appid
    JOIN genres gen ON gg.genre_id = gen.genre_id
    JOIN monthly_metrics m ON g.appid = m.appid
    GROUP BY g.appid, gen.name
    LIMIT 10
""", conn)

games: 380
game_genres: 1106
monthly_metrics: 27549


,name,release_date,genre,months_tracked
0,Grand Theft Auto IV: The Complete Edition,"Mar 24, 2020",Action,168
1,Grand Theft Auto IV: The Complete Edition,"Mar 24, 2020",Adventure,168
2,Space Engineers,"Feb 28, 2019",Action,153
3,Space Engineers,"Feb 28, 2019",Indie,153
4,Space Engineers,"Feb 28, 2019",Simulation,153
5,Space Engineers,"Feb 28, 2019",Strategy,153
6,Mount & Blade II: Bannerlord,"Oct 25, 2022",Action,76
7,Mount & Blade II: Bannerlord,"Oct 25, 2022",Indie,76
8,Mount & Blade II: Bannerlord,"Oct 25, 2022",RPG,76
9,Mount & Blade II: Bannerlord,"Oct 25, 2022",Simulation,76


In [9]:
frozen_appids = set(pd.read_csv("cohort_appids.csv")["AppID"])
db_appids = set(pd.read_sql("SELECT appid FROM games", conn)["appid"])
print("Match:",  frozen_appids == db_appids)


Match: True
